In [1]:
import threading
import time

# ... após o último print ...
print("Processo concluído.")

# Aguardar threads terminarem
time.sleep(1)

# Se ainda assim não funcionar, force o exit
import os
os._exit(0)

Processo concluído.


: 

In [ ]:
import os
import fitz
import uuid
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# -------------------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------------------
PASTA_PDFS = r"C:\Users\Yuan\Desktop\git_hub\ai_engineering\llmops"
MODELO_EMBEDDING = "./bge-small-en-v1.5"
TAMANHO_CHUNK = 450
SOBREPOSICAO = 60
COLECAO = "documentos_llm"
DIMENSAO = 384  # bge-small-en-v1.5

# -------------------------------------------------------------------------
# FUNÇÕES
# -------------------------------------------------------------------------

def extrair_texto_do_pdf(caminho_pdf):
    try:
        doc = fitz.open(caminho_pdf)
        texto = ""
        for pagina in doc:
            texto += pagina.get_text("text") + "\n"
        doc.close()
        return texto
    except Exception as e:
        print(f"Erro ao abrir {os.path.basename(caminho_pdf)} → {e}")
        return ""


def dividir_em_chunks(texto, tamanho=450, overlap=60):
    if not texto.strip():
        return []
    chunks = []
    inicio = 0
    while inicio < len(texto):
        fim = min(inicio + tamanho, len(texto))
        chunks.append(texto[inicio:fim])
        inicio += tamanho - overlap
    return chunks


def criar_colecao_se_nao_existir(client):
    colecoes = [c.name for c in client.get_collections().collections]
    if COLECAO not in colecoes:
        client.create_collection(
            collection_name=COLECAO,
            vectors_config=VectorParams(
                size=DIMENSAO,
                distance=Distance.COSINE
            )
        )
        print("Coleção criada.")
    else:
        print("Coleção já existe.")


# -------------------------------------------------------------------------
# MAIN
# -------------------------------------------------------------------------

if __name__ == "__main__":

    print("Carregando modelo...")
    modelo = SentenceTransformer(MODELO_EMBEDDING)
    print("Modelo carregado.\n")

    # 🔥 Qdrant EMBEDDED (SEM SERVIDOR)
    client = QdrantClient(path="./qdrant_data")

    criar_colecao_se_nao_existir(client)

    arquivos_pdf = [
        os.path.join(PASTA_PDFS, f)
        for f in os.listdir(PASTA_PDFS)
        if f.lower().endswith(".pdf")
    ]

    if not arquivos_pdf:
        print("Nenhum PDF encontrado.")
        exit()

    for caminho_pdf in tqdm(arquivos_pdf, desc="Processando PDFs"):

        nome = os.path.basename(caminho_pdf)
        texto = extrair_texto_do_pdf(caminho_pdf)
        chunks = dividir_em_chunks(texto, TAMANHO_CHUNK, SOBREPOSICAO)

        if not chunks:
            continue

        embeddings = modelo.encode(
            chunks,
            normalize_embeddings=True,
            batch_size=32
        )

        pontos = []
        for chunk, embedding in zip(chunks, embeddings):
            pontos.append(
                PointStruct(
                    id=str(uuid.uuid4()),
                    vector=embedding.tolist(),
                    payload={
                        "fonte": nome,
                        "texto": chunk
                    }
                )
            )

        client.upsert(
            collection_name=COLECAO,
            points=pontos
        )

    print("\nEmbeddings salvos localmente em ./qdrant_data 🚀")

    # ---------------------------------------------------------------------
    # TESTE DE BUSCA
    # ---------------------------------------------------------------------

    pergunta = "What is LLMOps?"
    embedding_query = modelo.encode(
        [pergunta],
        normalize_embeddings=True
    )[0]

    resultados = client.search(
        collection_name=COLECAO,
        query_vector=embedding_query.tolist(),
        limit=3
    )

    print("\nResultados da busca:\n")

    for r in resultados:
        print("Score:", r.score)
        print("Fonte:", r.payload["fonte"])
        print("Texto:", r.payload["texto"][:200])
        print("-" * 50)

In [ ]:
#from concurrent.futures import process

# -------------------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------------------
#MODELO_EMBEDDING = "./bge-small-en-v1.5"
#TAMANHO_CHUNK = 450
#SOBREPOSICAO = 60
#COLECAO = "documentos_llm"
#DIMENSAO = 384  # bge-small-en-v1.5


# -------------------------------------------------------------------------
# FUNÇÕES
# -------------------------------------------------------------------------

import os
import fitz
import uuid
from pathlib import Path
from tqdm import tqdm
import pytesseract
from pdf2image import convert_from_path
from process_glob import process

# -------------------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------------------
TAMANHO_CHUNK = 450
SOBREPOSICAO = 60

# -------------------------------------------------------------------------
# OCR (inicializa uma vez só)
# -------------------------------------------------------------------------
print("Inicializando OCR...")
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# -------------------------------------------------------------------------
# EXTRAÇÃO INTELIGENTE
# -------------------------------------------------------------------------

def extrair_texto_do_pdf(caminho_pdf):
    try:
        doc = fitz.open(caminho_pdf)

        # "Pegue o que já existe em texto_final e some com o texto da página"
        texto_final = ""

        for numero_pagina, pagina in enumerate(doc):
            texto = pagina.get_text("text").strip()

            # Se página tiver pouco texto, aplica OCR
            if len(texto) < 5:
                print(f"Página {numero_pagina+1} com pouco texto → aplicando OCR")

                imagens = convert_from_path(
                    caminho_pdf,
                    first_page=numero_pagina + 1,
                    last_page=numero_pagina + 1,
                    dpi=150
                )

                texto_ocr = pytesseract.image_to_string(imagens[0], lang="por")
                print("dasdasda")
                texto_final += texto_ocr + "\n"
            else:
                texto_final += texto + "\n"

        doc.close()
        return texto_final

    except Exception as e:
        print(f"Erro ao abrir {os.path.basename(caminho_pdf)} → {e}")
        return ""

# -------------------------------------------------------------------------
# CHUNKING
# -------------------------------------------------------------------------

def dividir_em_chunks(texto, tamanho=TAMANHO_CHUNK, overlap=SOBREPOSICAO):
    if not texto.strip():
        return []

    chunks = []
    inicio = 0

    while inicio < len(texto):
        fim = min(inicio + tamanho, len(texto))
        chunks.append(texto[inicio:fim])
        inicio += tamanho - overlap

    return chunks

# -------------------------------------------------------------------------
# PIPELINE PRINCIPAL
# -------------------------------------------------------------------------

def total_chunks():
    total_arquivos = 0
    total_chunks_count = 0
    todos_chunks = []

    for caminho_arquivo in process("pdf"):
        print(f"\nProcessando: {caminho_arquivo}")

        texto = extrair_texto_do_pdf(str(caminho_arquivo))

        if not texto:
            continue

        chunks = dividir_em_chunks(texto)

        print(f"Total de chunks gerados: {len(chunks)}")

        todos_chunks.extend(chunks)
        total_chunks_count += len(chunks)
        total_arquivos += 1

    print(f"\nArquivos processados: {total_arquivos}")
    print(f"Total de chunks: {total_chunks_count}")

    return todos_chunks


if __name__ == "__main__":
    total_chunks()
    """
    Execução direta do script.

    Uso:
      python extract_text_from_pdf.py [caminho_pdf]

    - Se nenhum argumento for passado, usa o valor de PASTA_PDFS.
    """

    #base_static = Path(__file__).resolve().parent.parent / "static" / "files"
'''
    if len(sys.argv) > 1:
        caminho_input = Path(sys.argv[1])
    else:
        caminho_input = Path(PASTA_PDFS)

    if not caminho_input.is_file():
        # tenta resolver dentro de enterprise_project/static/files
        candidato = base_static / caminho_input.name
        if candidato.is_file():
            caminho = candidato
        else:
            print(f"Arquivo não encontrado: {caminho_input}")
            print(f"Verifique se o PDF está em: {base_static}")
            print("Ou passe o caminho completo como argumento.")
            sys.exit(1)
    else:
        caminho = caminho_input

    print(f"Lendo PDF: {caminho}")
    texto = extrair_texto_do_pdf(str(caminho))

    if not texto:
        sys.exit(1)

    chunks = dividir_em_chunks(texto, tamanho=TAMANHO_CHUNK, overlap=SOBREPOSICAO)
    print(f"Total de caracteres: {len(texto)}")
    print(f"Total de chunks gerados: {len(chunks)}")
'''

# Qwen: Qwen3.5-35B-A3B <REDACTED_OPENROUTER_API_KEY>
# Qwen: Qwen3.5-35B-A3B <REDACTED_OPENROUTER_API_KEY>

In [ ]:
# enterprise_project/scripts/build_knowledge_base.py

import os
import glob
import uuid
from typing import List

# --- Ferramentas de Processamento de Documentos ---
import fitz  # PyMuPDF
from pdf2image import convert_from_path
import easyocr
import numpy as np

# --- Ferramentas de RAG (Chunking, Embedding, Indexing) ---
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

# ==============================================================================
# 1. CONFIGURAÇÃO (AJUSTE CONFORME NECESSÁRIO)
# ==============================================================================

# --- Caminhos e Pastas ---
# Caminho para o utilitário Poppler (essencial para pdf2image)
# SUBSTITUA PELO CAMINHO CORRETO NA SUA MÁQUINA!
POPPLER_PATH = r"C:\poppler\poppler-23.11.0\Library\bin"

# Diretório onde seus PDFs de origem estão localizados.
# O script vai procurar por todos os arquivos .pdf dentro desta pasta.
# O caminho é relativo à localização deste script.
SOURCE_DOCUMENTS_PATH = os.path.join(os.path.dirname(__file__), '..', 'static', 'files')

# --- Configuração do RAG ---
# Modelo de embedding. 'paraphrase-multilingual-MiniLM-L12-v2' é bom para múltiplos idiomas.
EMBEDDING_MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# --- Configuração do Qdrant (Modo Embedded) ---
# Caminho para a pasta onde a base de dados vetorial local será armazenada.
QDRANT_STORAGE_PATH = os.path.join(os.path.dirname(__file__), '..', 'qdrant_db')
QDRANT_COLLECTION_NAME = "knowledge_base_pdfs"


# ==============================================================================
# 2. FUNÇÕES DO PIPELINE
# ==============================================================================

def extract_text_from_pdf(pdf_path: str, ocr_reader: easyocr.Reader) -> str:
    """Extrai texto de um PDF, usando OCR com EasyOCR para páginas baseadas em imagem."""
    text_content = ""
    try:
        with fitz.open(pdf_path) as doc:
            for page_num, page in enumerate(doc):
                native_text = page.get_text("text").strip()
                if len(native_text) < 50:  # Limiar baixo indica página escaneada
                    print(f"   -> Página {page_num+1} com pouco texto nativo, aplicando OCR...")
                    images = convert_from_path(
                        pdf_path,
                        first_page=page_num + 1,
                        last_page=page_num + 1,
                        poppler_path=POPPLER_PATH,
                        dpi=200
                    )
                    if images:
                        ocr_result = ocr_reader.readtext(np.array(images[0]), detail=0, paragraph=True)
                        text_content += "\n".join(ocr_result) + "\n"
                else:
                    text_content += native_text + "\n"
    except Exception as e:
        print(f"❌ Erro ao extrair texto de {os.path.basename(pdf_path)}: {e}")
        return ""
    return text_content

def split_text_into_chunks(text: str) -> List[str]:
    """Divide o texto em pedaços (chunks) para embedding."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len
    )
    return text_splitter.split_text(text)

def create_and_store_embeddings(chunks: List[str], source_filename: str, model: SentenceTransformer, qdrant_client: QdrantClient):
    """Cria embeddings e os armazena no Qdrant."""
    embeddings = model.encode(chunks, show_progress_bar=False)
    
    qdrant_client.upsert(
        collection_name=QDRANT_COLLECTION_NAME,
        points=[
            models.PointStruct(
                id=str(uuid.uuid4()),
                vector=embedding.tolist(),
                payload={"text": chunk, "source": source_filename}
            )
            for chunk, embedding in zip(chunks, embeddings)
        ],
        wait=True
    )

# ==============================================================================
# 3. FUNÇÃO PRINCIPAL DE ORQUESTRAÇÃO
# ==============================================================================

def main():
    """Função principal que orquestra todo o pipeline de ingestão de documentos."""
    print("="*50)
    print("🚀 INICIANDO PIPELINE DE CONSTRUÇÃO DA BASE DE CONHECIMENTO 🚀")
    print("="*50)

    # --- Inicialização ---
    print("1. Inicializando modelos e clientes...")
    # Carrega o modelo de embedding uma vez
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    # Carrega o modelo de OCR uma vez
    ocr_reader = easyocr.Reader(['pt', 'en'])
    # Inicializa o cliente Qdrant em modo embedded
    qdrant_client = QdrantClient(path=QDRANT_STORAGE_PATH)

    # --- Criação da Coleção no Qdrant (se não existir) ---
    try:
        qdrant_client.get_collection(collection_name=QDRANT_COLLECTION_NAME)
        print(f"   -> Coleção '{QDRANT_COLLECTION_NAME}' já existe.")
    except Exception:
        print(f"   -> Criando nova coleção: '{QDRANT_COLLECTION_NAME}'")
        qdrant_client.create_collection(
            collection_name=QDRANT_COLLECTION_NAME,
            vectors_config=models.VectorParams(
                size=embedding_model.get_sentence_embedding_dimension(),
                distance=models.Distance.COSINE
            )
        )
    
    # --- Processamento dos Arquivos ---
    print(f"\n2. Procurando por arquivos PDF em: {SOURCE_DOCUMENTS_PATH}")
    pdf_files = glob.glob(f"{SOURCE_DOCUMENTS_PATH}/*.pdf")
    
    if not pdf_files:
        print("❌ Nenhum arquivo PDF encontrado. Verifique o caminho em SOURCE_DOCUMENTS_PATH.")
        return

    print(f"   -> Encontrados {len(pdf_files)} arquivos para processar.")

    for i, pdf_path in enumerate(pdf_files):
        filename = os.path.basename(pdf_path)
        print(f"\n--- Processando arquivo {i+1}/{len(pdf_files)}: {filename} ---")
        
        # Passo A: Extração de Texto
        print("   -> Extraindo texto...")
        full_text = extract_text_from_pdf(pdf_path, ocr_reader)
        if not full_text.strip():
            print("   -> ⚠️ Nenhum texto extraído. Pulando para o próximo arquivo.")
            continue
        
        # Passo B: Divisão em Chunks
        print("   -> Dividindo em chunks...")
        text_chunks = split_text_into_chunks(full_text)
        if not text_chunks:
            print("   -> ⚠️ Nenhum chunk gerado. Pulando para o próximo arquivo.")
            continue
        print(f"   -> Texto dividido em {len(text_chunks)} chunks.")
        
        # Passo C: Criação e Armazenamento dos Embeddings
        print("   -> Gerando embeddings e salvando no Qdrant...")
        create_and_store_embeddings(text_chunks, filename, embedding_model, qdrant_client)
        print(f"   -> ✅ '{filename}' processado e armazenado com sucesso.")

    print("\n="*50)
    print("🎉 PIPELINE CONCLUÍDO! Sua base de conhecimento foi atualizada. 🎉")
    print("="*50)

# ==============================================================================
# 4. PONTO DE ENTRADA DO SCRIPT
# ==============================================================================

if __name__ == "__main__":
    main()


In [ ]:
import os
from qdrant_client import QdrantClient

QDRANT_STORAGE_PATH = r'C:\Users\Yuan\Desktop\git_hub\ai_engineering\llmops\project_llmops\enterprise_project\qdrant_db'


client = QdrantClient(path=QDRANT_STORAGE_PATH)

collections = client.get_collections()
    
print("\n📦 Coleções encontradas:")
for collection in collections.collections:
    print(f"- {collection.name}")


: 

In [ ]:
points, next_page = client.scroll(
    collection_name="knowledge_base_pdfs",
    limit=5,              # quantos quer ler
    with_payload=True,    # trazer metadados
    with_vectors=False    # se quiser ver o embedding, coloque True
)

for p in points:
    print("ID:", p.id)
    print("Payload:", p.payload)
    print("-" * 50)

ID: 00e5adff-da56-4c7e-91a2-38240887a919
Payload: {'text': 'difference is that in Sn1, occurs firstly the ring opening polymerization (ROP) which will produce an \nopening chain cationic monomer (electrophile) that will be attacked by Y atom (nucleophile) of \nneighbor monomer and so on producing active point of growing chain until its formation in a polymeric \nchain, while that Sn2 has firstly a formation of cationic monomers cyclic which is attacked by neighbor \nmonomers [8].\nThe cationic ring-opening polymerization of lactide is induced many times for using organic acid \ncatalysts, but this paper it was replaced by ionic liquids. The reaction gets starting with activation of \nmonomer for protonation and subsequently the attack of carbonyl group of the initiator alcohol in \nlactide ring, and then forming lactil alcohol. The deprotonation of lactil alcohol cause the breaking of \nring and consequently, the group terminal hydroxyl will react with lactide monomers and inducing the

In [ ]:
points, _ = client.scroll(
    collection_name="knowledge_base_pdfs",
    limit=1,
    with_payload=True,
    with_vectors=True
)

print(points[0].vector)

[-0.03663643077015877, -0.05820152536034584, -0.020174706354737282, -0.02180752158164978, -0.024771368131041527, -0.006997117307037115, 0.019184686243534088, 0.05836772546172142, -0.01553762424737215, -0.059850599616765976, 0.03952164947986603, 0.03777756914496422, 0.03477539122104645, 0.028682736679911613, 0.03817654401063919, -0.09144989401102066, -0.05764410272240639, -0.04162470996379852, -0.0831853374838829, 0.07100703567266464, 0.011946849524974823, -0.04218102991580963, -0.01462708692997694, 0.007970952428877354, 0.07318558543920517, 0.0684790164232254, 0.02092236466705799, 0.05107598751783371, -0.04214034602046013, -0.26914432644844055, 0.03232210874557495, 0.05158894136548042, -0.010479726828634739, -0.009140309877693653, -0.010316168889403343, -0.047717008739709854, 0.0051260171458125114, 0.04266315698623657, -0.05570074915885925, -0.005637656897306442, -0.0023548761382699013, 0.034802697598934174, -0.024005556479096413, 0.00036387130967341363, -0.0007966056582517922, 0.02228

: 

In [ ]:
"""
Module: simple_query.py
======================

Script simples para consultar o banco de dados RAG.
"""

import os
from sentence_transformers import SentenceTransformer
from vector_store import initialize_qdrant_client, search_qdrant

# Carregar modelo de embeddings
print("Carregando modelo de embeddings...")
try:
    # Tentar carregar o modelo local
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
    PROJECT_DIR = os.path.dirname(os.path.dirname(SCRIPT_DIR))
    model_path = os.path.join(PROJECT_DIR, "model", "bge-small-en-v1.5")
    model = SentenceTransformer(model_path)
except:
    # Fallback para modelo online
    print("Usando modelo online alternativo...")
    model = SentenceTransformer("all-MiniLM-L6-v2")

# Conectar ao Qdrant
print("Conectando ao banco de dados...")
client = initialize_qdrant_client()

# Consulta
query = input("Digite sua consulta: ")
print(f"\nBuscando por: '{query}'")

# Gerar embedding e buscar
query_embedding = model.encode(query).tolist()
results = search_qdrant(client, query_embedding, top_k=5)

# Mostrar resultados
print(f"\nEncontrados {len(results)} resultados:")
print("-" * 50)

# Rastrear quais documentos foram encontrados
found_docs = set()

for i, result in enumerate(results):
    doc_name = result.get("source", "Desconhecido")
    found_docs.add(doc_name)
    
    print(f"\n{i+1}. Documento: {doc_name}")
    print(f"   Similaridade: {result['score']:.4f}")
    print(f"   Trecho: {result['text'][:200]}...")

print("\n" + "=" * 50)
print(f"Documentos relevantes encontrados: {len(found_docs)}")
for doc in found_docs:
    print(f"- {doc}")

In [ ]:
"""
Module: simple_query.py
======================

Script simples para consultar o banco de dados RAG.
"""

import os
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
import sys

# Adicionar o diretório pai ao path para importar o módulo vector_store
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_DIR = os.path.dirname(os.path.dirname(SCRIPT_DIR))
sys.path.append(os.path.join(PROJECT_DIR, "scripts"))
sys.path.append(os.path.join(PROJECT_DIR, "scripts", "rag"))

# Agora importamos as funções necessárias
from vector_store import initialize_qdrant_client

# Configuração do Qdrant
QDRANT_STORAGE_PATH = os.path.join(PROJECT_DIR, "qdrant_db")
QDRANT_COLLECTION_NAME = "knowledge_base_pdfs"

# Carregar modelo de embeddings
print("Carregando modelo de embeddings...")
try:
    # Tentar carregar o modelo local
    model_path = os.path.join(PROJECT_DIR, "model", "bge-small-en-v1.5")
    model = SentenceTransformer(model_path)
    print(f"Modelo carregado de: {model_path}")
except Exception as e:
    # Fallback para modelo online
    print(f"Erro ao carregar modelo local: {e}")
    print("Usando modelo online alternativo...")
    model = SentenceTransformer("all-MiniLM-L6-v2")

# Conectar ao Qdrant
print("Conectando ao banco de dados...")
try:
    client = QdrantClient(path=QDRANT_STORAGE_PATH)
    print(f"Conectado ao Qdrant em: {QDRANT_STORAGE_PATH}")
except Exception as e:
    print(f"Erro ao conectar ao Qdrant: {e}")
    sys.exit(1)

# Consulta
query = input("Digite sua consulta: ")
print(f"\nBuscando por: '{query}'")

# Gerar embedding e buscar
try:
    query_embedding = model.encode(query).tolist()
    
    # Usar a API correta do Qdrant Client
    search_results = client.search(
        collection_name=QDRANT_COLLECTION_NAME,
        query_vector=query_embedding,
        limit=5
    )
    
    # Converter resultados para o formato esperado
    results = []
    for result in search_results:
        # Extrair payload e adicionar score
        payload = result.payload
        payload["score"] = result.score
        results.append(payload)
    
except Exception as e:
    print(f"Erro durante a busca: {e}")
    sys.exit(1)

# Mostrar resultados
print(f"\nEncontrados {len(results)} resultados:")
print("-" * 50)

# Rastrear quais documentos foram encontrados
found_docs = set()

for i, result in enumerate(results):
    doc_name = result.get("source", "Desconhecido")
    found_docs.add(doc_name)
    
    print(f"\n{i+1}. Documento: {doc_name}")
    print(f"   Similaridade: {result['score']:.4f}")
    print(f"   Trecho: {result['text'][:200]}...")

print("\n" + "=" * 50)
print(f"Documentos relevantes encontrados: {len(found_docs)}")
for doc in found_docs:
    print(f"- {doc}")